In [7]:
from semanticscholar import SemanticScholar
from habanero import Crossref
from itertools import product
import pandas as pd
import json


In [8]:
# search params
primary_keywords = ['Generative AI']  # and so on...
secondary_keywords = ['Social Impact']  # and so on...
tertiary_keywords = ['Education']  # and so on...

year = "2019-"

all_combinations = list(product(primary_keywords, secondary_keywords, tertiary_keywords))

In [9]:
# Prepare DataFrame to store the results
columns = [
    'PaperTitle',
    'DOI',
    'Authors',
    'Abstract',
    'Publisher',
    'SemanticScholarUrl',
    'DoiUrl',
    'PublicationDate',
    'FieldOfStudy',
    'Conference-Journal',
    'PublicationTypes',
    'SearchString',
    'CitationCount',
    'SearchedFrom'
]
results_df = pd.DataFrame(columns=columns)

In [10]:


from requests import HTTPError

sch = SemanticScholar()
crossref = Crossref()
paper_count = 0
for combination in all_combinations:
    query = " AND ".join(combination)
    
    # Search using the combined query
    try:
        sch_results = sch.search_paper(query, year=year)
        for sch_paper in sch_results:
            try:
                crossref_paper = crossref.works(ids=sch_paper['externalIds'].get('DOI'))
            except HTTPError as e:
                print(f"An HTTP error occurred for query '{query}': {e}, error code: {e.response.status_code}")
                crossref_paper = None
            except Exception as e:
                print(f"An error occurred for query '{query}': {e.with_traceback()}")
                crossref_paper = None
         
            title = sch_paper['title']
            doi = sch_paper['externalIds'].get('DOI')
            
            # author name and affiliation
            if crossref_paper is not None:
                authors = crossref_paper.get("message").get("author")
                for i in range(len(authors)):
                    author = authors[i]
                    author_name = author.get("given", "") + " " + author.get("family", "")
                    affiliation = author.get("affiliation", "No Affiliation")
                    affiliations = author.get("affiliation", [])
                    school_names = [affil.get("name") for affil in affiliations] if affiliations else ["No Affiliation"]
                    # Create a new dictionary with only 'name' and 'affiliation'
                    authors[i] = {
                        "name": author_name.strip(),
                        "affiliation": school_names
                    }
            else:
                authors = sch_paper['authors']
                for i in range(len(sch_paper['authors'])):
                    author = sch_paper['authors'][i]
                    sch_paper['authors'][i] = {
                        "name": author.get("name", "No Name"),
                        "affiliation": author.get("affiliation", "No Affiliation")
                    }
                
            abstract = sch_paper['abstract']
            sch_url = sch_paper['url']
            doi_url = f"https://doi.org/{sch_paper['externalIds']['DOI']}"
            publication_date = sch_paper['publicationDate']
            fields_of_study = sch_paper['fieldsOfStudy']
            venue = sch_paper['venue']
            
            # publisher
            if crossref_paper is not None:   
                publisher = crossref_paper.get("message").get("publisher")
            elif "arxiv" in doi.lower():
                publisher = "arXiv"
            else:
                publisher = None
                
            # paper type
            if crossref_paper is not None:
                paper_type = [crossref_paper.get("message").get("type")]
            else:
                paper_type = sch_paper['publicationTypes']
                
            citation_count = sch_paper['citationCount']
            # TODO: keywords missing
            # TODO: paper type is conference/journal for arxiv papers
            # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.
            
            
            new_paper = {
                'PaperTitle': title,
                'DOI': doi,
                'Authors': authors,
                'Abstract': abstract,
                'Publisher': publisher,
                'SemanticScholarUrl': sch_url,
                'DoiUrl': doi_url,
                'PublicationDate': publication_date,
                'FieldOfStudy': fields_of_study,
                'Conference-Journal': venue,
                'PublicationTypes': paper_type,
                'SearchString': query,
                'CitationCount': citation_count,
                'SearchedFrom': 'Semantic Scholar',
            }
            
            print(f"----- Paper {paper_count}: {title} -------\n")
            print(json.dumps(new_paper, indent=2))
            print("\n\n")
            results_df = pd.concat([results_df, pd.DataFrame([new_paper])], ignore_index=True) 
            paper_count += 1
            
            # TODO for testing
            if paper_count >= 10:
                break
            
    except Exception as e:
        results_df.to_csv('data/initial-scrape-result.csv', index=False)
        print(f"An error occurred for query '{query}': {e}")
        
    results_df.to_csv('data/initial-scrape-result.csv', index=False)

----- Paper 0: The Social Impact of Generative AI: An Analysis on ChatGPT -------

{
  "PaperTitle": "The Social Impact of Generative AI: An Analysis on ChatGPT",
  "DOI": "10.1145/3582515.3609555",
  "Authors": [
    {
      "name": "Maria Teresa Baldassarre",
      "affiliation": [
        "Computer Science, University of Bari, Italy"
      ]
    },
    {
      "name": "Danilo Caivano",
      "affiliation": [
        "Computer Science, University of Bari, Italy"
      ]
    },
    {
      "name": "Berenice Fernandez Nieto",
      "affiliation": [
        "Computer Science, University of Bari, Italy"
      ]
    },
    {
      "name": "Domenico Gigante",
      "affiliation": [
        "Computer Science, University of Bari, Italy"
      ]
    },
    {
      "name": "Azzurra Ragone",
      "affiliation": [
        "Computer Science, University of Bari, Italy"
      ]
    }
  ],
  "Abstract": "In recent months, the impact of Artificial Intelligence (AI) on citizens\u2019 lives has gained

In [13]:
# semantic scholar test
sch = SemanticScholar()
sch_results = sch.get_paper("10.1177/030631284014003004")


In [12]:
#corssref test

cr = Crossref()
cr_results = cr.works(query = "Education and Information Technologies : Official Journal of the IFIP technical committee on Education")